<a href="https://colab.research.google.com/github/jintubhuyan-2000/Spatial-Gradient-of-Highway-Induced-Land-Cover-Change/blob/main/SECTION_4_10_%E2%80%94_VEGETATION_FRAGMENTATION_AND_CORE_VEGETATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =============================================================================
# SECTION 4.10
# VEGETATION FRAGMENTATION AND CORE VEGETATION — 2025
# =============================================================================
#
# PURPOSE:
#   Publication-ready statistical analysis of:
#
#   1. Vegetation patch structure
#   2. Patch-size distribution
#   3. Small vegetation patches
#   4. Core vegetation
#
# INPUTS EXPECTED:
#
#   TZPR_NLP_Vegetation_PatchSize_2025.tif
#   TZPR_NLP_Small_Vegetation_Patches_2025.tif
#   TZPR_NLP_Core_Vegetation_2025.tif
#
# The script automatically searches for these files.
#
# OUTPUTS:
#
#   01_Vegetation_Fragmentation_2025_Statistics.csv
#   02_Vegetation_Patch_Size_Distribution_2025.csv
#   03_Small_Vegetation_Patches_2025.csv
#   04_Core_Vegetation_2025.csv
#   05_Publication_Table_4_10.csv
#   06_Section_4_10_Summary.txt
#
# =============================================================================


import os
import glob
import math
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage


# =============================================================================
# 1. USER INPUT FOLDER
# =============================================================================

INPUT_DIR = r"/content/drive/MyDrive/TZPR_NLP_Research"

OUTPUT_DIR = os.path.join(
    INPUT_DIR,
    "Vegetation_Fragmentation_Analysis_4_10"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 2. AUTOMATIC FILE SEARCH
# =============================================================================

def find_raster(folder, keywords):

    """
    Find a raster whose filename contains all supplied keywords.
    """

    extensions = ["*.tif", "*.TIF", "*.tiff", "*.TIFF"]

    files = []

    for ext in extensions:
        files.extend(
            glob.glob(
                os.path.join(folder, ext)
            )
        )

    matches = []

    for f in files:

        name = os.path.basename(f).lower()

        if all(
            keyword.lower() in name
            for keyword in keywords
        ):
            matches.append(f)

    if len(matches) == 0:
        return None

    # Prefer exact expected naming
    matches = sorted(
        matches,
        key=lambda x: len(os.path.basename(x))
    )

    return matches[0]


# =============================================================================
# 3. FIND INPUT RASTERS
# =============================================================================

PATCH_RASTER = find_raster(
    INPUT_DIR,
    ["vegetation", "patch", "size", "2025"]
)

SMALL_RASTER = find_raster(
    INPUT_DIR,
    ["small", "vegetation", "patch", "2025"]
)

CORE_RASTER = find_raster(
    INPUT_DIR,
    ["core", "vegetation", "2025"]
)


print("\n")
print("=" * 85)
print("VEGETATION FRAGMENTATION — INPUT FILE DETECTION")
print("=" * 85)


print("\nVegetation Patch Size:")
print(
    PATCH_RASTER
    if PATCH_RASTER
    else "NOT FOUND"
)

print("\nSmall Vegetation Patches:")
print(
    SMALL_RASTER
    if SMALL_RASTER
    else "NOT FOUND"
)

print("\nCore Vegetation:")
print(
    CORE_RASTER
    if CORE_RASTER
    else "NOT FOUND"
)


# =============================================================================
# 4. CHECK CORE RASTER
# =============================================================================

if CORE_RASTER is None:

    raise FileNotFoundError(
        "\n\nCore vegetation raster was not found.\n"
        "Expected something similar to:\n"
        "TZPR_NLP_Core_Vegetation_2025.tif\n"
    )


# =============================================================================
# 5. READ RASTER
# =============================================================================

def read_raster(path):

    with rasterio.open(path) as src:

        data = src.read(1)

        nodata = src.nodata

        if nodata is not None:

            valid = data != nodata

        else:

            valid = np.ones(
                data.shape,
                dtype=bool
            )

        valid &= np.isfinite(data)

        profile = src.profile.copy()

        transform = src.transform

        crs = src.crs

        width = src.width

        height = src.height

        bounds = src.bounds

    return (
        data,
        valid,
        profile,
        transform,
        crs,
        width,
        height,
        bounds
    )


# =============================================================================
# 6. PIXEL AREA
# =============================================================================

def calculate_pixel_area_ha(
    transform,
    width,
    height,
    crs
):

    """
    Calculates pixel area in hectares.

    EPSG:4326:
        Uses latitude-dependent geographic pixel area.

    Projected CRS:
        Uses affine pixel dimensions.
    """

    if crs is not None and crs.is_geographic:

        R = 6378137.0

        dlon = abs(transform.a)
        dlat = abs(transform.e)

        top_lat = transform.f

        row_area = np.zeros(height)

        for row in range(height):

            lat_top = (
                top_lat -
                row * dlat
            )

            lat_bottom = (
                lat_top -
                dlat
            )

            phi1 = math.radians(
                lat_bottom
            )

            phi2 = math.radians(
                lat_top
            )

            area_m2 = (
                R ** 2
                *
                math.radians(dlon)
                *
                (
                    math.sin(phi2)
                    -
                    math.sin(phi1)
                )
            )

            row_area[row] = (
                area_m2 / 10000.0
            )

        return np.repeat(
            row_area[:, None],
            width,
            axis=1
        )

    else:

        pixel_width = abs(
            transform.a
        )

        pixel_height = abs(
            transform.e
        )

        pixel_area_ha = (
            pixel_width *
            pixel_height /
            10000.0
        )

        return np.full(
            (height, width),
            pixel_area_ha
        )


# =============================================================================
# 7. CREATE VEGETATION MASK
# =============================================================================

def vegetation_mask(data, valid):

    """
    Any valid value > 0 is treated as vegetation.

    This works for:
        binary vegetation masks
        0/1 masks
        categorical vegetation masks
        patch-ID rasters
    """

    return (
        valid &
        (data > 0)
    )


# =============================================================================
# 8. IDENTIFY CONNECTED PATCHES
# =============================================================================

def identify_patches(mask):

    """

    Uses 8-neighbour connectivity.

    Diagonal vegetation pixels are therefore
    considered part of the same patch.
    """

    structure = np.ones(
        (3, 3),
        dtype=np.uint8
    )

    labels, number = ndimage.label(
        mask,
        structure=structure
    )

    return labels, number


# =============================================================================
# 9. PATCH AREA CALCULATION
# =============================================================================

def patch_area_statistics(
    mask,
    pixel_area_ha
):

    labels, number = identify_patches(
        mask
    )

    if number == 0:

        return (
            np.array([]),
            labels
        )

    label_values = labels[mask]

    area_values = pixel_area_ha[mask]

    patch_areas = np.bincount(
        label_values,
        weights=area_values,
        minlength=number + 1
    )[1:]

    patch_areas = patch_areas[
        patch_areas > 0
    ]

    return (
        patch_areas,
        labels
    )


# =============================================================================
# 10. READ CORE RASTER
# =============================================================================

print("\n")
print("=" * 85)
print("READING CORE VEGETATION RASTER")
print("=" * 85)


core_data, core_valid, core_profile, core_transform, core_crs, core_width, core_height, core_bounds = read_raster(
    CORE_RASTER
)


core_mask = vegetation_mask(
    core_data,
    core_valid
)


# =============================================================================
# 11. PIXEL AREA
# =============================================================================

pixel_area_ha = calculate_pixel_area_ha(
    core_transform,
    core_width,
    core_height,
    core_crs
)


# =============================================================================
# 12. CORE VEGETATION AREA
# =============================================================================

core_area_ha = pixel_area_ha[
    core_mask
].sum()


core_area_km2 = (
    core_area_ha /
    100.0
)


print(
    f"\nCore vegetation area:"
    f" {core_area_ha:,.2f} ha"
)

print(
    f"Core vegetation area:"
    f" {core_area_km2:,.2f} km²"
)


# =============================================================================
# 13. IF PATCH-SIZE RASTER EXISTS
# =============================================================================

if PATCH_RASTER is not None:

    print("\n")
    print("=" * 85)
    print("PROCESSING VEGETATION PATCH STRUCTURE")
    print("=" * 85)

    patch_data, patch_valid, patch_profile, patch_transform, patch_crs, patch_width, patch_height, patch_bounds = read_raster(
        PATCH_RASTER
    )

    patch_mask = vegetation_mask(
        patch_data,
        patch_valid
    )

    patch_pixel_area = calculate_pixel_area_ha(
        patch_transform,
        patch_width,
        patch_height,
        patch_crs
    )

    patch_areas, patch_labels = (
        patch_area_statistics(
            patch_mask,
            patch_pixel_area
        )
    )

else:

    print(
        "\nWARNING:"
        "\nVegetation Patch Size raster not found."
    )

    patch_areas = np.array([])


# =============================================================================
# 14. VEGETATION PATCH STATISTICS
# =============================================================================

if len(patch_areas) > 0:

    vegetation_area_ha = (
        patch_areas.sum()
    )

    number_patches = (
        len(patch_areas)
    )

    mean_patch = (
        np.mean(patch_areas)
    )

    median_patch = (
        np.median(patch_areas)
    )

    minimum_patch = (
        np.min(patch_areas)
    )

    maximum_patch = (
        np.max(patch_areas)
    )

    std_patch = (
        np.std(
            patch_areas,
            ddof=1
        )
        if number_patches > 1
        else 0
    )

else:

    vegetation_area_ha = 0
    number_patches = 0
    mean_patch = 0
    median_patch = 0
    minimum_patch = 0
    maximum_patch = 0
    std_patch = 0


# =============================================================================
# 15. PATCH SIZE CLASSES
# =============================================================================

patch_classes = [

    (
        "Very small",
        0,
        1
    ),

    (
        "Small",
        1,
        5
    ),

    (
        "Medium",
        5,
        25
    ),

    (
        "Large",
        25,
        100
    ),

    (
        "Very large",
        100,
        np.inf
    )

]


distribution_rows = []


for class_name, lower, upper in patch_classes:

    if upper == np.inf:

        selected = (
            patch_areas >= lower
        )

    else:

        selected = (
            (patch_areas >= lower)
            &
            (patch_areas < upper)
        )

    selected_areas = (
        patch_areas[selected]
    )

    number = len(
        selected_areas
    )

    area = (
        selected_areas.sum()
    )

    percentage = (
        area /
        vegetation_area_ha *
        100
        if vegetation_area_ha > 0
        else 0
    )

    distribution_rows.append({

        "Patch_Size_Class":
            class_name,

        "Lower_Limit_ha":
            lower,

        "Upper_Limit_ha":
            upper
            if upper != np.inf
            else "No upper limit",

        "Number_of_Patches":
            number,

        "Area_ha":
            area,

        "Area_km2":
            area / 100.0,

        "Percentage_of_Vegetation":
            percentage
    })


distribution_df = pd.DataFrame(
    distribution_rows
)


# =============================================================================
# 16. SMALL PATCH ANALYSIS
# =============================================================================

# Definition:
# Small vegetation patches = <5 ha

small_patches = (
    patch_areas < 5
)

small_areas = (
    patch_areas[
        small_patches
    ]
)

small_number = (
    len(small_areas)
)

small_area_ha = (
    small_areas.sum()
)

small_area_km2 = (
    small_area_ha /
    100.0
)


small_patch_percentage = (

    small_number /
    number_patches *
    100

    if number_patches > 0
    else 0
)


small_area_percentage = (

    small_area_ha /
    vegetation_area_ha *
    100

    if vegetation_area_ha > 0
    else 0
)


# =============================================================================
# 17. CORE PATCH STRUCTURE
# =============================================================================

core_patch_areas, core_labels = (
    patch_area_statistics(
        core_mask,
        pixel_area_ha
    )
)


number_core_patches = (
    len(core_patch_areas)
)


if number_core_patches > 0:

    mean_core = (
        np.mean(
            core_patch_areas
        )
    )

    median_core = (
        np.median(
            core_patch_areas
        )
    )

    minimum_core = (
        np.min(
            core_patch_areas
        )
    )

    maximum_core = (
        np.max(
            core_patch_areas
        )
    )

else:

    mean_core = 0
    median_core = 0
    minimum_core = 0
    maximum_core = 0


# =============================================================================
# 18. CORE VEGETATION PERCENTAGE
# =============================================================================

if vegetation_area_ha > 0:

    core_percentage = (

        core_area_ha /
        vegetation_area_ha *
        100

    )

else:

    core_percentage = 0


# =============================================================================
# 19. PUBLICATION TABLE
# =============================================================================

publication_data = {

    "Year":
        2025,

    "Total_Vegetation_Area_ha":
        vegetation_area_ha,

    "Total_Vegetation_Area_km2":
        vegetation_area_ha / 100.0,

    "Number_of_Vegetation_Patches":
        number_patches,

    "Mean_Patch_Area_ha":
        mean_patch,

    "Median_Patch_Area_ha":
        median_patch,

    "Minimum_Patch_Area_ha":
        minimum_patch,

    "Maximum_Patch_Area_ha":
        maximum_patch,

    "SD_Patch_Area_ha":
        std_patch,

    "Very_Small_Patches_Number":
        distribution_df.loc[
            distribution_df[
                "Patch_Size_Class"
            ] == "Very small",
            "Number_of_Patches"
        ].iloc[0]
        if len(distribution_df) > 0
        else 0,

    "Small_Patches_lt5ha_Number":
        small_number,

    "Small_Patches_lt5ha_Area_ha":
        small_area_ha,

    "Small_Patches_lt5ha_Area_km2":
        small_area_km2,

    "Small_Patches_Percentage_of_Patches":
        small_patch_percentage,

    "Small_Patches_Percentage_of_Vegetation":
        small_area_percentage,

    "Core_Vegetation_Area_ha":
        core_area_ha,

    "Core_Vegetation_Area_km2":
        core_area_km2,

    "Core_Vegetation_Percentage":
        core_percentage,

    "Number_of_Core_Patches":
        number_core_patches,

    "Mean_Core_Patch_Area_ha":
        mean_core,

    "Median_Core_Patch_Area_ha":
        median_core,

    "Minimum_Core_Patch_Area_ha":
        minimum_core,

    "Maximum_Core_Patch_Area_ha":
        maximum_core
}


publication_df = pd.DataFrame(
    [publication_data]
)


# =============================================================================
# 20. SAVE OUTPUTS
# =============================================================================

statistics_path = os.path.join(
    OUTPUT_DIR,
    "01_Vegetation_Fragmentation_2025_Statistics.csv"
)

distribution_path = os.path.join(
    OUTPUT_DIR,
    "02_Vegetation_Patch_Size_Distribution_2025.csv"
)

small_path = os.path.join(
    OUTPUT_DIR,
    "03_Small_Vegetation_Patches_2025.csv"
)

core_path = os.path.join(
    OUTPUT_DIR,
    "04_Core_Vegetation_2025.csv"
)

publication_path = os.path.join(
    OUTPUT_DIR,
    "05_Publication_Table_4_10.csv"
)


# Full vegetation statistics
pd.DataFrame([
    {
        "Metric":
            "Total vegetation area (ha)",
        "Value":
            vegetation_area_ha
    },
    {
        "Metric":
            "Number of vegetation patches",
        "Value":
            number_patches
    },
    {
        "Metric":
            "Mean patch area (ha)",
        "Value":
            mean_patch
    },
    {
        "Metric":
            "Median patch area (ha)",
        "Value":
            median_patch
    },
    {
        "Metric":
            "Minimum patch area (ha)",
        "Value":
            minimum_patch
    },
    {
        "Metric":
            "Maximum patch area (ha)",
        "Value":
            maximum_patch
    },
    {
        "Metric":
            "Standard deviation of patch area (ha)",
        "Value":
            std_patch
    }
]).to_csv(
    statistics_path,
    index=False
)


# Patch distribution
distribution_df.to_csv(
    distribution_path,
    index=False
)


# Small patches
pd.DataFrame([
    {
        "Metric":
            "Small patch threshold",
        "Value":
            "< 5 ha"
    },
    {
        "Metric":
            "Number of small patches",
        "Value":
            small_number
    },
    {
        "Metric":
            "Small patch area (ha)",
        "Value":
            small_area_ha
    },
    {
        "Metric":
            "Small patch area (km2)",
        "Value":
            small_area_km2
    },
    {
        "Metric":
            "Percentage of all patches",
        "Value":
            small_patch_percentage
    },
    {
        "Metric":
            "Percentage of vegetation area",
        "Value":
            small_area_percentage
    }
]).to_csv(
    small_path,
    index=False
)


# Core vegetation
pd.DataFrame([
    {
        "Metric":
            "Core vegetation area (ha)",
        "Value":
            core_area_ha
    },
    {
        "Metric":
            "Core vegetation area (km2)",
        "Value":
            core_area_km2
    },
    {
        "Metric":
            "Core vegetation percentage",
        "Value":
            core_percentage
    },
    {
        "Metric":
            "Number of core patches",
        "Value":
            number_core_patches
    },
    {
        "Metric":
            "Mean core patch area (ha)",
        "Value":
            mean_core
    },
    {
        "Metric":
            "Median core patch area (ha)",
        "Value":
            median_core
    },
    {
        "Metric":
            "Minimum core patch area (ha)",
        "Value":
            minimum_core
    },
    {
        "Metric":
            "Maximum core patch area (ha)",
        "Value":
            maximum_core
    }
]).to_csv(
    core_path,
    index=False
)


# Publication table
publication_df.to_csv(
    publication_path,
    index=False
)


# =============================================================================
# 21. MANUSCRIPT SUMMARY
# =============================================================================

summary_path = os.path.join(
    OUTPUT_DIR,
    "06_Section_4_10_Summary.txt"
)


with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "SECTION 4.10 — VEGETATION FRAGMENTATION "
        "AND CORE VEGETATION\n"
    )

    f.write(
        "=" * 80 + "\n\n"
    )

    f.write(
        "2025 VEGETATION PATCH STRUCTURE\n"
    )

    f.write(
        f"Total vegetation area: "
        f"{vegetation_area_ha:,.2f} ha\n"
    )

    f.write(
        f"Total vegetation area: "
        f"{vegetation_area_ha / 100.0:,.2f} km2\n"
    )

    f.write(
        f"Number of vegetation patches: "
        f"{number_patches:,}\n"
    )

    f.write(
        f"Mean patch area: "
        f"{mean_patch:,.2f} ha\n"
    )

    f.write(
        f"Median patch area: "
        f"{median_patch:,.2f} ha\n"
    )

    f.write(
        f"Largest vegetation patch: "
        f"{maximum_patch:,.2f} ha\n"
    )

    f.write(
        "\nSMALL VEGETATION PATCHES (<5 ha)\n"
    )

    f.write(
        f"Number: "
        f"{small_number:,}\n"
    )

    f.write(
        f"Area: "
        f"{small_area_ha:,.2f} ha\n"
    )

    f.write(
        f"Percentage of all patches: "
        f"{small_patch_percentage:.2f}%\n"
    )

    f.write(
        f"Percentage of vegetation area: "
        f"{small_area_percentage:.2f}%\n"
    )

    f.write(
        "\nCORE VEGETATION\n"
    )

    f.write(
        f"Core vegetation area: "
        f"{core_area_ha:,.2f} ha\n"
    )

    f.write(
        f"Core vegetation area: "
        f"{core_area_km2:,.2f} km2\n"
    )

    f.write(
        f"Core vegetation percentage: "
        f"{core_percentage:.2f}%\n"
    )

    f.write(
        f"Number of core patches: "
        f"{number_core_patches:,}\n"
    )

    f.write(
        f"Mean core patch area: "
        f"{mean_core:,.2f} ha\n"
    )

    f.write(
        f"Median core patch area: "
        f"{median_core:,.2f} ha\n"
    )

    f.write(
        f"Largest core patch: "
        f"{maximum_core:,.2f} ha\n"
    )


# =============================================================================
# 22. FINAL CONSOLE OUTPUT
# =============================================================================

print("\n")
print("=" * 85)
print("PUBLICATION-READY VEGETATION FRAGMENTATION RESULTS")
print("=" * 85)


print(
    f"\nTotal vegetation area:"
    f" {vegetation_area_ha:,.2f} ha"
)

print(
    f"Number of vegetation patches:"
    f" {number_patches:,}"
)

print(
    f"Mean patch area:"
    f" {mean_patch:,.2f} ha"
)

print(
    f"Median patch area:"
    f" {median_patch:,.2f} ha"
)

print(
    f"Largest patch:"
    f" {maximum_patch:,.2f} ha"
)


print(
    f"\nSmall patches (<5 ha):"
    f" {small_number:,}"
)

print(
    f"Small-patch area:"
    f" {small_area_ha:,.2f} ha"
)

print(
    f"Small-patch percentage of vegetation:"
    f" {small_area_percentage:.2f}%"
)


print(
    f"\nCore vegetation:"
    f" {core_area_ha:,.2f} ha"
)

print(
    f"Core vegetation percentage:"
    f" {core_percentage:.2f}%"
)

print(
    f"Number of core patches:"
    f" {number_core_patches:,}"
)

print(
    f"Largest core patch:"
    f" {maximum_core:,.2f} ha"
)


print("\n")
print("=" * 85)
print("OUTPUT FILES")
print("=" * 85)

print(
    f"\n{statistics_path}"
)

print(
    distribution_path
)

print(
    small_path
)

print(
    core_path
)

print(
    publication_path
)

print(
    summary_path
)

print("\nAnalysis completed successfully.")
print("=" * 85)



VEGETATION FRAGMENTATION — INPUT FILE DETECTION

Vegetation Patch Size:
/content/drive/MyDrive/TZPR_NLP_Research/TZPR_NLP_Vegetation_PatchSize_2025.tif

Small Vegetation Patches:
/content/drive/MyDrive/TZPR_NLP_Research/TZPR_NLP_Small_Vegetation_Patches_2025.tif

Core Vegetation:
/content/drive/MyDrive/TZPR_NLP_Research/TZPR_NLP_Core_Vegetation_2025.tif


READING CORE VEGETATION RASTER

Core vegetation area: 180,339.76 ha
Core vegetation area: 1,803.40 km²


PROCESSING VEGETATION PATCH STRUCTURE


PUBLICATION-READY VEGETATION FRAGMENTATION RESULTS

Total vegetation area: 210,356.05 ha
Number of vegetation patches: 3,740
Mean patch area: 56.24 ha
Median patch area: 0.13 ha
Largest patch: 50,554.83 ha

Small patches (<5 ha): 3,373
Small-patch area: 1,783.37 ha
Small-patch percentage of vegetation: 0.85%

Core vegetation: 180,339.76 ha
Core vegetation percentage: 85.73%
Number of core patches: 3,424
Largest core patch: 42,199.34 ha


OUTPUT FILES

/content/drive/MyDrive/TZPR_NLP_Researc

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
